In [1]:
from pathlib import Path
import sys

# Google Colab provides this module; a normal local Python environment does not.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab asks you to authorize access to your own Google Drive.
    drive.mount('/content/drive')
    # Upload the NeuroVisionAI project folder here, including its src folder.
    ROOT = Path('/content/drive/MyDrive/NeuroVisionAI')
else:
    # Keep local execution available for editing and small checks.
    ROOT = Path.cwd()
    if not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent

if not (ROOT / 'src' / 'introduction_to_deeplearning').is_dir():
    raise FileNotFoundError(
        f'Missing helper code: {ROOT / "src"}. '
        'Upload the complete NeuroVisionAI folder to MyDrive, not just the notebook.'
    )
# Python can now import our commented helper functions from the uploaded src folder.
sys.path.insert(0, str(ROOT / 'src'))
print('Project folder:', ROOT)


Project folder: c:\Users\Youcode\Documents\DEV_AI\sprint_1\Introduction_to_deeplearning


In [ ]:
import hashlib
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# ROOT is your Drive project folder on Colab, so results survive a runtime reset.
PROCESSED_DATA_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
from introduction_to_deeplearning.data import load_split, make_dataset
from introduction_to_deeplearning.model import build_cnn, compile_model
from introduction_to_deeplearning.evaluate import plot_history, evaluate_split, plot_examples
from introduction_to_deeplearning.explain import make_gradcam
from introduction_to_deeplearning.predict import predict_image


## 2. Choose one experiment
First run: learning rate 0.0003, augmentation on, original architecture.
Next compare augmentation off. Then compare extra convolution on, keeping everything
else unchanged. Compare **validation** macro-F1, loss and recall per class.

30 is a maximum, not a requirement: early stopping will end an unproductive run.
A fixed seed reduces random variation, but different hardware can still affect results.


In [ ]:
EXPERIMENT_NAME = 'baseline_aug'
PROCESSED_PATH = PROCESSED_DATA_DIR
SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 0.0003
EPOCHS = 30
USE_AUGMENTATION = True
EXTRA_CONVOLUTION = False
RESUME_RUN_DIR = None  # Example: ROOT / 'models' / 'YOUR_INTERRUPTED_RUN'
RUN_SMALL_CHECK = False  # Optional diagnostic in section 5, not the final model.

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if IN_COLAB and not gpus:
    raise RuntimeError('Enable a GPU in Runtime > Change runtime type, then rerun the setup cells.')
print('Reading saved splits from:', PROCESSED_PATH)
print('Saving models and reports to:', MODELS_DIR)


## 3. Load and verify training and validation only
The old run used 4,617 training / 990 validation images; the older notebook output
showed 4,916 / 1,053. Print the actual sizes each time instead of relying on old output.
The saved arrays must be uint8: dividing already normalized arrays again would be wrong.


In [ ]:
# Run preprocessing.ipynb first, or upload your existing processed arrays to this folder.
if not (PROCESSED_PATH / 'preprocessing_config.json').is_file():
    raise FileNotFoundError(f'No saved splits in {PROCESSED_PATH}. Run preprocessing.ipynb first.')
X_train, y_train = load_split(PROCESSED_PATH, 'train')
X_val, y_val = load_split(PROCESSED_PATH, 'val')
config = json.loads((PROCESSED_PATH / 'preprocessing_config.json').read_text(encoding='utf-8'))
class_names = config['class_names']
if class_names != ['glioma', 'meningioma', 'notumor', 'pituitary']:
    raise ValueError('Unexpected class order. Check the preprocessing configuration.')
print('Training:', X_train.shape, '| validation:', X_val.shape)
print('Training batches:', int(np.ceil(len(X_train) / BATCH_SIZE)))
print('Class order:', class_names)
print('Split limitation:', config.get('split_unit', 'Not documented'))


## 4. Check duplicates and label alignment
We hash training and validation inputs to catch exact overlap in an older export.
This does not detect near-duplicates or shared patients. Do not inspect test images
while experimenting. Its overlap check will happen during final evaluation.


In [ ]:
def image_hashes(images):
    # One signature per image: identical resized pixels produce the same signature.
    return [hashlib.sha256(image.tobytes()).hexdigest() for image in images]

train_hashes = image_hashes(X_train)
val_hashes = image_hashes(X_val)
if len(set(train_hashes)) != len(train_hashes) or len(set(val_hashes)) != len(val_hashes):
    raise ValueError('Duplicates found within a split. Regenerate clean splits before training.')
if set(train_hashes) & set(val_hashes):
    raise ValueError('Training/validation overlap found. Fix the splits before training.')
print('No identical training/validation images overlap.')

for split, labels in [('train', y_train), ('validation', y_val)]:
    print(split, dict(zip(class_names, labels.sum(axis=0).astype(int).tolist())))


## 5. Optional: can the CNN memorize 32 images?
This is a pipeline diagnostic, not a performance estimate. Select eight training
images per class, turn augmentation and dropout off, and train a fresh small-check
model. It should usually learn these examples well. If accuracy stays low, inspect
images, labels, normalization and optimization before doing larger experiments.
The diagnostic model is discarded; the real experiment creates a new model.


In [ ]:
if RUN_SMALL_CHECK:
    selected = []
    for class_index in range(4):
        selected.extend(np.flatnonzero(y_train.argmax(axis=1) == class_index)[:8])
    small_ds = make_dataset(X_train[selected], y_train[selected], training=True,
                            augment=False, batch_size=8, seed=SEED)
    small_model = build_cnn(dropout=0.0)
    compile_model(small_model, learning_rate=0.001)
    small_model.fit(small_ds, epochs=60, verbose=2, shuffle=False,
                    callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor='accuracy', mode='max', baseline=0.99,
                        patience=15, restore_best_weights=True)])
    print('Small-check metrics:', small_model.evaluate(small_ds, return_dict=True))
    del small_model
else:
    print('Skipped. Set RUN_SMALL_CHECK=True to run this diagnostic.')


## 6. Build datasets and inspect augmentation
Only training is shuffled or augmented. All images are divided by 255 once in
`make_dataset`; the CNN must not contain a second Rescaling layer.
Small rotations, zoom and shifts create slightly different examples each epoch.
Inspect the examples below to check that their content remains recognizable.


In [ ]:
train_ds = make_dataset(X_train, y_train, training=True, augment=USE_AUGMENTATION,
                        batch_size=BATCH_SIZE, seed=SEED)
val_ds = make_dataset(X_val, y_val, batch_size=BATCH_SIZE)
preview_ds = make_dataset(X_train[:4], y_train[:4], training=True,
                          augment=USE_AUGMENTATION, batch_size=4, seed=SEED)
preview_images, preview_labels = next(iter(preview_ds))
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(preview_images[i, ..., 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(class_names[int(tf.argmax(preview_labels[i]))])
    ax.axis('off')
plt.show()

# Recalculate weights from this training split instead of reusing old JSON weights.
counts = y_train.sum(axis=0)
class_weights = None
if counts.max() / counts.min() >= 1.5:
    weights = compute_class_weight('balanced', classes=np.arange(4), y=y_train.argmax(axis=1))
    class_weights = dict(enumerate(weights.tolist()))
print('Class weights:', class_weights)


## 7. Create a fresh CNN
Convolutions learn patterns, pooling reduces spatial size, and global average
pooling keeps the parameter count small. Dense combines features. Dropout reduces
reliance on individual neurons. Softmax produces four class scores that sum to one.
See `model.py` for the commented architecture. Adam uses the explicit learning rate.


In [ ]:
# Reset again so the optional diagnostic cannot change the real model initialization.
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
model = build_cnn(extra_convolution=EXTRA_CONVOLUTION, dropout=0.5)
compile_model(model, LEARNING_RATE)
model.summary()

if RESUME_RUN_DIR is None:
    RUN_DIR = MODELS_DIR / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '_' + EXPERIMENT_NAME)
    RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    RUN_DIR = Path(RESUME_RUN_DIR)
    if not (RUN_DIR / 'backup').is_dir():
        raise ValueError('No interrupted-run backup found in this folder.')
# Record the augmentation actually used, replacing any older export defaults.
config['augmentation'] = {'enabled': USE_AUGMENTATION, 'rotation': 0.03,
                          'zoom': 0.05, 'translation': 0.05}
settings = {'experiment': EXPERIMENT_NAME, 'seed': SEED, 'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE, 'epochs': EPOCHS,
            'augmentation': USE_AUGMENTATION, 'extra_convolution': EXTRA_CONVOLUTION,
            'dropout': 0.5, 'tensorflow_version': tf.__version__,
            'processed_path': str(PROCESSED_PATH.resolve()),
            'train_size': len(X_train), 'validation_size': len(X_val),
            'train_fingerprint': hashlib.sha256(''.join(train_hashes).encode() + y_train.tobytes()).hexdigest(),
            'validation_fingerprint': hashlib.sha256(''.join(val_hashes).encode() + y_val.tobytes()).hexdigest()}
# Refuse to resume if settings or data changed since the interrupted run.
if RESUME_RUN_DIR is not None:
    previous_settings = json.loads((RUN_DIR / 'settings.json').read_text(encoding='utf-8'))
    for key in ['seed', 'batch_size', 'learning_rate', 'epochs', 'augmentation',
                'extra_convolution', 'dropout', 'train_fingerprint', 'validation_fingerprint']:
        if previous_settings[key] != settings[key]:
            raise ValueError(f'Resume settings do not match: {key}')
# Save class order alongside the model so inference never guesses label names.
(RUN_DIR / 'preprocessing_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(RUN_DIR / 'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
summary_lines = []
model.summary(print_fn=summary_lines.append)
(RUN_DIR / 'architecture.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
try:
    tf.keras.utils.plot_model(model, to_file=str(RUN_DIR / 'architecture.png'), show_shapes=True)
except (ImportError, OSError) as error:
    print('Optional architecture picture needs Graphviz; the text summary is saved.', error)
print('Run folder:', RUN_DIR)


## 8. Train and save progress
- ModelCheckpoint saves the lowest-validation-loss model to disk.
- BackupAndRestore saves progress after each epoch for interruption recovery.
- ReduceLROnPlateau lowers the learning rate after two unproductive epochs.
- EarlyStopping waits six unproductive epochs, giving the lower rate time to help.
- CSVLogger saves each epoch even if the session is interrupted.

To resume an interrupted run, set RESUME_RUN_DIR in section 2 to its existing
folder, keep the same settings and data, then run sections in order again.
BackupAndRestore restores completed epoch progress; callback patience counters
restart in a new Python session. Keep EPOCHS unchanged when resuming.


In [ ]:
# When resuming, protect the best checkpoint from earlier completed epochs.
previous_best = None
if (RUN_DIR / 'epochs.csv').exists():
    previous_best = float(pd.read_csv(RUN_DIR / 'epochs.csv')['val_loss'].min())
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(RUN_DIR / 'best.keras'), monitor='val_loss',
                                       save_best_only=True, initial_value_threshold=previous_best),
    tf.keras.callbacks.BackupAndRestore(str(RUN_DIR / 'backup')),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.CSVLogger(str(RUN_DIR / 'epochs.csv'), append=True),
]
start = time.perf_counter()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weights, callbacks=callbacks, verbose=2, shuffle=False)
training_seconds = time.perf_counter() - start
settings['training_seconds'] = training_seconds
(RUN_DIR / 'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
(RUN_DIR / 'history.json').write_text(json.dumps(history.history, indent=2), encoding='utf-8')
print(f'Training time: {training_seconds / 60:.1f} minutes')
# Reload the saved best model so evaluation and the application use the same weights.
model = tf.keras.models.load_model(RUN_DIR / 'best.keras')


## 9. Interpret learning curves
If training loss falls while validation loss rises, the model is overfitting.
If both stay high, investigate optimization, features or data quality.
Training accuracy includes augmentation and dropout, whereas validation does not,
so their difference is not a perfect measurement of overfitting on its own.


In [ ]:
plot_history(history, RUN_DIR)
plt.show()


## 10. Validate and inspect mistakes
Look beyond accuracy: macro-F1 treats all classes equally. Recall tells us how many
actual examples of each class were found. Watch pituitary recall and the number
of tumor images called notumor. Record observations in docs/experiments.csv.
Example indices correspond to rows of val_manifest.csv when that manifest exists.


In [ ]:
val_scores, _ = evaluate_split(model, X_val, y_val, class_names, RUN_DIR)
plt.show()
plot_examples(X_val, y_val, val_scores, class_names, RUN_DIR)
plt.show()


## 11. Explain a validation prediction with Grad-CAM
The heatmap shows locations influencing the predicted class score. It does not
outline a tumor and cannot establish that the model learned medically useful features.
Inspect whether the model emphasizes anatomy, borders or text annotations.
The gradient calculation is isolated in explain.py because it is optional to the CNN lesson.


In [ ]:
wrong = np.flatnonzero(val_scores.argmax(axis=1) != y_val.argmax(axis=1))
example_index = int(wrong[0]) if len(wrong) else 0
heatmap = make_gradcam(X_val[example_index], model)
fig, ax = plt.subplots()
ax.imshow(X_val[example_index, ..., 0], cmap='gray')
ax.imshow(heatmap, cmap='jet', alpha=0.4, vmin=0, vmax=1)
ax.set_title(f'Validation index {example_index}: {class_names[val_scores[example_index].argmax()]}')
ax.axis('off')
fig.savefig(RUN_DIR / 'validation_gradcam.png')
plt.show()


## 12. Final test: opt in only after choosing the model
Leave this off while comparing experiments. Select the run using validation,
then set FINAL_RUN_DIR to that run folder and RUN_FINAL_TEST=True.
This cell loads its saved best model and rejects a different dataset fingerprint.
A saved test report prevents an accidental repeated evaluation of that run.
If you already used test results to select models, document that limitation: a flag
cannot turn a previously examined test set back into an independent holdout.


In [ ]:
RUN_FINAL_TEST = False
FINAL_RUN_DIR = RUN_DIR  # Or Path('absolute/path/to/the/chosen/run')
if RUN_FINAL_TEST:
    if (FINAL_RUN_DIR / 'test_report.json').exists():
        raise ValueError('This run already has a test report. Read it instead of evaluating again.')
    final_settings = json.loads((FINAL_RUN_DIR / 'settings.json').read_text(encoding='utf-8'))
    for key in ['train_fingerprint', 'validation_fingerprint']:
        if final_settings[key] != settings[key]:
            raise ValueError('The selected model used different data. Restore its processed dataset first.')
    final_config = json.loads((FINAL_RUN_DIR / 'preprocessing_config.json').read_text(encoding='utf-8'))
    X_test, y_test = load_split(PROCESSED_PATH, 'test')
    test_hashes = image_hashes(X_test)
    if len(set(test_hashes)) != len(test_hashes) or set(test_hashes) & (set(train_hashes) | set(val_hashes)):
        raise ValueError('Duplicate test images found. The split needs review.')
    final_model = tf.keras.models.load_model(FINAL_RUN_DIR / 'best.keras')
    test_scores, _ = evaluate_split(final_model, X_test, y_test, final_config['class_names'], FINAL_RUN_DIR, 'test')
    plt.show()
    plot_examples(X_test, y_test, test_scores, final_config['class_names'], FINAL_RUN_DIR, 'test')
    plt.show()
else:
    print('Test skipped: keep it reserved until model selection is finished.')


## 13. Predict one file and open the demo
`predict_image` uses the same grayscale conversion, resize and normalization as training.
Choose a real image path below. Unknown or unreadable files produce a clear error.
In Colab, the cell below can predict a Drive image directly. The Streamlit demo
is for local use after downloading the chosen model folder; it is optional for training.


In [ ]:
IMAGE_PATH = None  # Example: Path('/content/drive/MyDrive/example_mri.jpg')
if IMAGE_PATH is not None:
    result = predict_image(IMAGE_PATH, RUN_DIR)
    print('Predicted:', result['class'])
    print('Scores:', result['scores'])
